## RAG EVALUATION (LangSmith)

Two things are evaluated separately, because a wrong answer can come from either stage:

| Stage | Question | Evaluators |
|---|---|---|
| **Retrieval** | Did we fetch the right documents? | `context_relevance` (LLM judge) |
| **Generation** | Is the answer good, given what was retrieved? | `correctness` (vs reference), `groundedness` (vs retrieved context), `answer_relevance` (vs question) |


In [3]:
import sys
sys.path.append('..')
from src.rag_pipeline import load_documents, build_hybrid_retriever, build_llm


d:\RAG_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
documents = load_documents()    
hybrid_retriever = build_hybrid_retriever()
llm = build_llm()

Loading embedding model: BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1773.30it/s]


Model loaded. Embedding dimension: 384
Vector store ready. Collection: json_qa_documents (2964 documents)
BM25 index built over 2964 documents


In [8]:
import json, random, re, os
import openai
from langsmith import Client, traceable, wrappers

os.environ["LANGSMITH_TRACING"] = "true"
assert os.getenv("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY missing from .env"

ls_client = Client()

# Judge client: Groq via its OpenAI-compatible endpoint (wrapped so judge calls are traced too)
judge_client = wrappers.wrap_openai(
    openai.OpenAI(
        api_key=os.getenv("EVAL_GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1",
    )
)
# Ideally use a DIFFERENT model from the generator (llm above) to avoid self-preference bias.
JUDGE_MODEL = "openai/gpt-oss-20b"

### 1. Build the evaluation dataset
The corpus is already Q&A pairs with a `doc_id`, so we can build a labelled set automatically:


In [12]:
DATASET_NAME = "RAG Eval - Mental Health QA"
N_EXAMPLES = 10
random.seed(42)

def strip_think(text: str) -> str:
    """Remove <think>...</think> blocks some reasoning models emit."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()

def split_qa(page_content: str):
    q, a = page_content.split("\nAnswer: ", 1)
    return q.removeprefix("Question: ").strip(), a.strip()

def paraphrase(question: str) -> str:
    r = judge_client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": "Rewrite the user's question the way a real person might phrase it, "
                                          "using different words but the same meaning. Output ONLY the rewritten question."},
            {"role": "user", "content": question},
        ],
    )
    return strip_think(r.choices[0].message.content)

def build_examples(docs):
    examples = []
    for d in docs:
        q, a = split_qa(d.page_content)
        examples.append({
            "inputs":  {"question": paraphrase(q)},
            "outputs": {"answer": a, "doc_id": d.metadata["doc_id"]},
            "metadata": {
                "topic_group": d.metadata.get("topic_group"),
                "safety_sensitive": "safety_sensitive" in (d.metadata.get("flags") or []),
                "original_question": q,
            },
        })
    return examples

if ls_client.has_dataset(dataset_name=DATASET_NAME):
    print("Dataset already exists, reusing it.")
else:
    dataset = ls_client.create_dataset(DATASET_NAME, description="Paraphrased questions with gold answer + gold doc_id")
    ls_client.create_examples(dataset_id=dataset.id, examples=build_examples(random.sample(documents, N_EXAMPLES)))
    print(f"Created dataset with {N_EXAMPLES} examples.")

Created dataset with 10 examples.


### Retrieve both docs and answer

In [14]:
def make_target(retriever, top_k: int = 3):
    
    @traceable(run_type="retriever", name="retrieve")
    def retrieve(query: str):
        docs = retriever.retrieve(query, top_k=top_k)
        return [
            {"page_content": d["content"], "type": "Document",
             "metadata": {"id": d["id"], "rank": d["rank"]}}
            for d in docs
        ]

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        docs = retrieve(question)
        context = "\n\n".join(d["page_content"] for d in docs)
        if not context:
            return {"response": "No relevant context found to answer the question.",
                    "retrieved_ids": [], "context": ""}

        prompt = (
            "Use the following context to answer the question concisely. "
            "Answer only from the context.\n\n"
            f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
        )
        response = llm.invoke(prompt).content
        return {
            "response": strip_think(response),
            "retrieved_ids": [d["metadata"]["id"] for d in docs],
            "context": context,
        }
    return target

### Evaluators or Metrics

1. **Correctness:** Response vs reference answer (`Does Answer Match the Groundtruth Answer?`)
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [15]:
from typing_extensions import Annotated, TypedDict

## Correctness Output Schema

## Grade output schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str,...,'Explain your reasoning for the score']
    correct: Annotated[bool,...,'True if the answer is correct, False otherwise.']

## correctness prompt
correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_groq import ChatGroq

grader_llm=ChatGroq(model=JUDGE_MODEL,temperature=0,groq_api_key=os.getenv('EVAL_GROQ_API_KEY'),).with_structured_output(CorrectnessGrade)

## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['response']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return {'key':'correctness', 'score':int(grade['correct']), 'comment':grade['explanation']}

**2. Answer Relevance:** Response vs input (Does answer address the question?)
- We simply look at the inputs and outputs without needing the reference_outputs.

In [ ]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""